# ⏱️ GIADA TG-01 — matrice temporale full-state
Studio supplementare fuori roadmap, inizialmente chiamato Task 10 per errore. La Task 10 originale (sufficienza dell'ingresso) resta aperta. Un solo run: acquisizione NEURON a 0,125 ms, quattro granularità interne, due seed appaiati, checkpoint multipli, rollout a 8 ms e costo GPU. L'uscita resta ogni 1 ms. Il risultato è uno screening causale della famiglia MLP, non la prova universale che un passo sia impossibile.


In [ ]:
from pathlib import Path
import base64, json, os, shutil, subprocess, sys, traceback
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_10');GIADA_REPO=WORK/'giada';TEACHER_REPO=WORK/'neuron_as_deep_net'
assert not GIADA_REPO.exists() and not TEACHER_REPO.exists(),'Sessione già inizializzata: usa un notebook Kaggle nuovo.'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip();print({'revision':REVISION})


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7','numpy','pandas','matplotlib','h5py','pyarrow','pyyaml'],check=True)
SIMULATION_DIR=TEACHER_REPO/'L5PC_NEURON_simulation'
if not list(SIMULATION_DIR.rglob('libnrnmech.so')):
 nrnivmodl=shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl')
 subprocess.run([nrnivmodl,'mods'],cwd=SIMULATION_DIR,check=True)
assert list(SIMULATION_DIR.rglob('libnrnmech.so')),'Compilazione MOD fallita'
import torch
assert torch.cuda.is_available(),'La matrice full-state richiede GPU CUDA Kaggle.'
print({'gpu':torch.cuda.get_device_name(0),'teacher_mods_compiled':True})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher.physiological_path_floor import load_verified_task9
from src.giada_teacher.temporal_granularity_matrix import acquire_full_state_matrix,train_paired_granularity_models
plan=json.loads((GIADA_REPO/'experiments/teacher_temporal_granularity_matrix_plan_v1.json').read_text())
display({'task':plan['task'],'internal_substeps':plan['internal_substeps_per_ms'],'seeds':plan['seeds'],'checkpoints':plan['checkpoint_steps'],'external_ms':plan['external_report_interval_ms']})


## 📁 Input necessari
Mantieni `hayflow-targeted-transition-dataset-v1-1-base` **completo di snapshots** e l'artefatto piccolo `giada_physiological_voltage_paths.zip` della Task 9. Non servono Task 9b o Task 9c come input; i loro risultati sono già registrati.


In [ ]:
INPUT_ROOT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK9_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
 candidates+=list(INPUT_ROOT.rglob('giada_physiological_voltage_paths.zip'))
 candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'physiological' in str(p).lower()]
 candidates += [p.parent for p in INPUT_ROOT.rglob('selected_paths.json') if (p.parent/'final_report.json').is_file()]
TASK9_SOURCE=None
for path in candidates:
 if not path.exists():continue
 try:load_verified_task9(path);TASK9_SOURCE=path.resolve();break
 except (RuntimeError,FileNotFoundError,ValueError,KeyError):continue
assert TASK9_SOURCE is not None,'Artefatto Task 9 esatto non trovato.'
base_override=os.environ.get('GIADA_TARGETED_DATASET')
base_candidates=[Path(base_override).expanduser()] if base_override else []
base_candidates += [Path('/kaggle/input/datasets/alessandrobelli/hayflow-targeted-transition-dataset-v1-1-base')]
if INPUT_ROOT.is_dir():base_candidates += [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower()]
BASE_ROOT=next((p.resolve() for p in base_candidates if p.is_dir() and all((p/name).is_file() for name in ('transition_dataset.h5','state_schema.json','dataset_manifest.json','synapses.parquet')) and (p/'snapshots').is_dir()),None)
assert BASE_ROOT is not None,'Dataset targeted v1.1 base completo non trovato: servono HDF5, manifest, schema, synapses.parquet e snapshots.'
print({'task9_source':str(TASK9_SOURCE),'base_root':str(BASE_ROOT)})


## 🔬 Acquisizione e replay integrale
Questa è la fase NEURON CPU. Stampa progressi ed ETA ogni dieci transizioni. Il training non parte se anche un solo confine o RNG non coincide.


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_temporal_granularity_matrix')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
def progress(percent,label):print(f'[GIADA TG-01][SHA-256 {label}] {percent}%',flush=True)
try:
 acquisition=acquire_full_state_matrix(BASE_ROOT,TASK9_SOURCE,TEACHER_REPO,GIADA_REPO,OUTPUT_DIR,progress=progress)
except Exception as exc:
 OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
 acquisition={'valid':False,'blocker':'acquisition_runtime_failure','error':repr(exc),'traceback_tail':traceback.format_exc()[-3000:]}
 (OUTPUT_DIR/'acquisition_report.json').write_text(json.dumps(acquisition,indent=2))
display({'valid':acquisition['valid'],'support':acquisition.get('selected_transition_count'),'roles':acquisition.get('role_counts'),'max_boundary_error':acquisition.get('maximum_boundary_error'),'max_micro_voltage_error':acquisition.get('maximum_micro_voltage_error_mv'),'max_rng_error':acquisition.get('maximum_rng_error'),'blocker':acquisition.get('blocker')})
RUN_TRAINING=bool(acquisition['valid'])


## 🎛️ Matrice GPU appaiata
Quattro granularità × due seed, 600 step ciascuno. Ogni braccio produce una transizione esterna da 1 ms. I checkpoint 100/300/600 vengono selezionati solo su development; test diagnostico e rollout sono aperti dopo il freeze.


In [ ]:
if RUN_TRAINING:
 try:
  report=train_paired_granularity_models(BASE_ROOT,OUTPUT_DIR,code_revision=REVISION)
 except Exception as exc:
  report={'schema_version':'giada-task10-temporal-granularity-matrix-v1','valid':False,'blocker':'training_runtime_failure','error':repr(exc),'traceback_tail':traceback.format_exc()[-3000:]}
  (OUTPUT_DIR/'final_report.json').write_text(json.dumps(report,indent=2))
else:report={'valid':False,'blocker':'teacher_replay_preflight'}
display({'valid':report['valid'],'decision':report.get('decision'),'selected_substeps':report.get('selected_internal_substeps_per_ms'),'rollout_windows':report.get('rollout_window_count'),'blocker':report.get('blocker')})


In [ ]:
if report['valid']:
 import pandas as pd
 display(pd.DataFrame(report['evaluations']))
 display(pd.DataFrame(report['rollout_rows']))
 print('Decisione:',report['decision'])
 if report['decision'].startswith('INCONCLUSIVE'):print('Nessuna conclusione sulla necessità di ridurre il passo: il canary non ha superato i gate di learnability/supporto.')
else:print('Risultato bloccato; leggere il report prima di interpretare la granularità temporale.')


## 📦 Scarica solo i report
Il file `training_support.npz` resta nell'output Kaggle ma non è inserito nello ZIP: il download diagnostico è piccolo. Metodo Blob/base64 concordato.


In [ ]:
EXPORT_DIR=Path('/kaggle/working/giada_temporal_granularity_matrix_report')
assert not EXPORT_DIR.exists(),'Export già presente; usa una sessione nuova.'
EXPORT_DIR.mkdir()
for name in ('acquisition_report.json','final_report.json'):
 source=OUTPUT_DIR/name
 if source.is_file():shutil.copy2(source,EXPORT_DIR/name)
archive=Path(shutil.make_archive('/kaggle/working/giada_temporal_granularity_matrix_report','zip',EXPORT_DIR.parent,EXPORT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
